In [ ]:
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# Project paths
PROJECT_ROOT = Path.cwd()

# If Jupyter was launched from /notebooks, move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "imagery"
GIS_DIR = PROJECT_ROOT / "data" / "raw" / "gis"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
GIS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Image directory:", IMAGE_DIR)

In [ ]:
PANORAMA_API = "https://api.data.amsterdam.nl/panorama/panoramas/"

# Test point around Weesperstraat
test_lon = 4.90775
test_lat = 52.36268

params = {
    "near": f"{test_lon},{test_lat}",
    "srid": 4326,
    "radius": 100,
    "page_size": 5,
}

response = requests.get(
    PANORAMA_API,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("Final URL:", response.url)
print(response.text[:1000])

In [ ]:
data = response.json()

panoramas = data["_embedded"]["panoramas"]

print("Returned in this page:", len(panoramas))
print("Total panoramas within radius:", data["count"])

# Inspect all fields available for the first panorama
first = panoramas[0]

for key, value in first.items():
    print(f"{key}: {value}")

In [ ]:
# Show a compact summary of the five returned panoramas
for i, pano in enumerate(panoramas):
    print(f"\n--- Panorama {i+1} ---")

    print("ID:", pano.get("pano_id"))
    print("Timestamp:", pano.get("timestamp"))
    print("Mission year:", pano.get("mission_year"))
    print("Geometry:", pano.get("geometry"))

    links = pano.get("_links", {})

    print(
        "Medium image:",
        links.get("equirectangular_medium", {}).get("href")
    )

In [ ]:
image_url = first["_links"]["equirectangular_medium"]["href"]

print(image_url)

img_response = requests.get(image_url, timeout=60)

print("Image status:", img_response.status_code)
print("Content type:", img_response.headers.get("Content-Type"))
print("Size:", len(img_response.content) / 1024 / 1024, "MB")

In [ ]:
img = Image.open(BytesIO(img_response.content))

print("Image dimensions:", img.size)

plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Amsterdam Panorama Test")
plt.show()

In [ ]:
test_image_path = IMAGE_DIR / "AMS_test_001.jpg"

img.save(test_image_path, quality=95)

print("Saved to:", test_image_path)

In [ ]:
# Query more panoramas around the test location
params_recent = {
    "near": f"{test_lon},{test_lat}",
    "srid": 4326,
    "radius": 30,       # tighter radius
    "page_size": 100,
}

r_recent = requests.get(
    PANORAMA_API,
    params=params_recent,
    timeout=30
)

r_recent.raise_for_status()

recent_data = r_recent.json()
recent_panos = recent_data["_embedded"]["panoramas"]

print("Returned:", len(recent_panos))
print("Total within 30 m:", recent_data["count"])

# Convert useful fields into a dataframe
records = []

for pano in recent_panos:
    coords = pano["geometry"]["coordinates"]

    records.append({
        "pano_id": pano["pano_id"],
        "timestamp": pano["timestamp"],
        "mission_year": pano["mission_year"],
        "longitude": coords[0],
        "latitude": coords[1],
        "heading": pano["heading"],
        "image_url": pano["_links"]["equirectangular_medium"]["href"],
    })

df_panos = pd.DataFrame(records)

df_panos["timestamp"] = pd.to_datetime(
    df_panos["timestamp"],
    utc=True,
    format="ISO8601"
)

df_panos = df_panos.sort_values(
    "timestamp",
    ascending=False
)

df_panos.head(15)

In [ ]:
print(
    df_panos["timestamp"].min(),
    "→",
    df_panos["timestamp"].max()
)

print()

print(
    df_panos["timestamp"]
    .dt.year
    .value_counts()
    .sort_index()
)

In [ ]:
%pip install py360convert

import numpy as np
import py360convert

# Convert PIL image to numpy
pano_np = np.array(img)

# Generate a conventional perspective view
perspective = py360convert.e2p(
    pano_np,
    fov_deg=90,
    u_deg=0,      # horizontal viewing direction
    v_deg=0,      # vertical viewing direction
    out_hw=(768, 768)
)

plt.figure(figsize=(8, 8))
plt.imshow(perspective)
plt.axis("off")
plt.title("Perspective view — 0°")
plt.show()

In [ ]:
headings = [0, 90, 180, 270]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, heading in zip(axes, headings):

    view = py360convert.e2p(
        pano_np,
        fov_deg=90,
        u_deg=heading,
        v_deg=0,
        out_hw=(512, 512)
    )

    ax.imshow(view)
    ax.set_title(f"{heading}°")
    ax.axis("off")

plt.tight_layout()
plt.show()